# 歌词采集

In [2]:
import requests
import re
import json
import os
import time
import pandas as pd
from collections import defaultdict

import jieba
import jieba.posseg as pseg
from collections import Counter
from openai import OpenAI

In [3]:
jieba.load_userdict('data/mayday_dict_simple.txt')

Building prefix dict from the default dictionary ...
Loading model from cache /var/folders/zj/y5ymt78d6qd4p13cf1q_3rch0000gn/T/jieba.cache
Loading model cost 0.527 seconds.
Prefix dict has been built successfully.


In [4]:
import sys
sys.path.append('..')

# 歌曲数据采集与清洗

# 原始数据清洗

In [5]:
df_raw = pd.read_excel('data/mayday_songs.xlsx', sheet_name='Sheet1')
df_raw['song_name'] = df_raw['song_name'].astype(str)
df_raw

,album_order,album_id,album_name,album_type,release_date,song_order,song_id,song_name,album_fixed,has_lyric,is_duplicate
0,1,38315,第一张创作专辑,录音室专辑,1999-07-07,1,386925,疯狂世界,第一张创作专辑,1,0
1,1,38315,第一张创作专辑,录音室专辑,1999-07-07,2,386927,拥抱,第一张创作专辑,1,0
2,1,38315,第一张创作专辑,录音室专辑,1999-07-07,3,386929,透露,第一张创作专辑,1,0
3,1,38315,第一张创作专辑,录音室专辑,1999-07-07,4,386930,生活,第一张创作专辑,1,0
4,1,38315,第一张创作专辑,录音室专辑,1999-07-07,5,386931,爱情的模样,第一张创作专辑,1,0
...,...,...,...,...,...,...,...,...,...,...,...
185,11,2740205,步步 自选作品辑,精选辑,2013-12-30,26,28181124,雌雄同体,步步 自选作品辑,1,1
186,11,2740205,步步 自选作品辑,精选辑,2013-12-30,27,28181126,生命有一种绝对,步步 自选作品辑,1,1
187,11,2740205,步步 自选作品辑,精选辑,2013-12-30,28,28181128,诺亚方舟,步步 自选作品辑,1,1
188,11,2740205,步步 自选作品辑,精选辑,2013-12-30,29,28181130,我心中尚未崩坏的地方,步步 自选作品辑,1,1


In [6]:
df_raw[['album_id', 'album_name']].drop_duplicates()

,album_id,album_name
0,38315,第一张创作专辑
12,38308,爱情万岁
24,38297,人生海海
36,38276,时光机
51,38259,神的孩子都在跳舞
64,38241,为爱而生
77,38235,后青春期的诗
89,2040001,第二人生 (明日版)
103,38214,第二人生 (末日版)
117,34746073,自传


In [7]:
df_raw['song_name'].value_counts()

song_name
干杯      3
憨人      3
诺亚方舟    3
三个傻瓜    3
星空      3
       ..
小护士     1
垃圾车     1
小时候     1
王子面     1
盛夏光年    1
Name: count, Length: 137, dtype: int64

In [24]:
# 清除song_name中的标点符号
# 完全抹除标点和其周围的所有空格
df_raw['song_name_fixed'] = df_raw['song_name'].str.replace(r'\s*[^\w\s]\s*|_', '', regex=True).str.strip()
df_unique = df_raw.drop_duplicates(subset='song_name_fixed', keep='first').copy()
df_unique['song_name'] = df_unique['song_name'].astype(str)
df_unique

,album_order,album_id,album_name,album_type,release_date,song_order,song_id,song_name,album_fixed,has_lyric,is_duplicate,song_name_fixed
0,1,38315,第一张创作专辑,录音室专辑,1999-07-07,1,386925,疯狂世界,第一张创作专辑,1,0,疯狂世界
1,1,38315,第一张创作专辑,录音室专辑,1999-07-07,2,386927,拥抱,第一张创作专辑,1,0,拥抱
2,1,38315,第一张创作专辑,录音室专辑,1999-07-07,3,386929,透露,第一张创作专辑,1,0,透露
3,1,38315,第一张创作专辑,录音室专辑,1999-07-07,4,386930,生活,第一张创作专辑,1,0,生活
4,1,38315,第一张创作专辑,录音室专辑,1999-07-07,5,386931,爱情的模样,第一张创作专辑,1,0,爱情的模样
...,...,...,...,...,...,...,...,...,...,...,...,...
163,11,2740205,步步 自选作品辑,精选辑,2013-12-30,4,28181109,洋葱 (2013新录制作品),步步 自选作品辑,1,0,洋葱2013新录制作品
165,11,2740205,步步 自选作品辑,精选辑,2013-12-30,6,28181113,入阵曲,步步 自选作品辑,1,0,入阵曲
169,11,2740205,步步 自选作品辑,精选辑,2013-12-30,10,28181121,离开地球表面,步步 自选作品辑,1,0,离开地球表面
175,11,2740205,步步 自选作品辑,精选辑,2013-12-30,16,28181104,你是唯一 (2013新录制作品),步步 自选作品辑,1,0,你是唯一2013新录制作品


In [25]:
df_unique['album_name'].value_counts()

album_name
时光机           15
第二人生 (明日版)    14
神的孩子都在跳舞      13
为爱而生          13
自传            13
第一张创作专辑       12
爱情万岁          12
人生海海          12
后青春期的诗        12
知足 最真杰作选      11
步步 自选作品辑       7
第二人生 (末日版)     2
Name: count, dtype: int64

# 歌词采集

In [ ]:
# --- 核心清洗函数 ---
def clean_and_format_lyrics(raw_text):
    if not raw_text or "未能获取" in raw_text:
        return ""

    # 1. 预处理：处理特殊空格 U+00A0
    text = raw_text.replace('\u00a0', ' ')

    # 2. 定义黑名单关键词
    exclude_keywords = [
        '作词', '作曲', '编曲', '：', ':', '演奏', '吉他', '贝斯',
        '鼓', '编写', '演唱', '合唱', '制作', '录音', '混音', 'ISRC',
        '编码', '版权', '提供', '发行', 'OP', 'SP'
    ]

    # 3. 专门针对 ISRC 这种特征码的正则表达式
    # 匹配规律：大写字母开头，中间有多个连字符和数字，例如 TW-K23-08-016-11
    isrc_pattern = r'[A-Z]{2}-[A-Z0-9]{3}-\d{2}-\d{5}'

    pure_lyrics_list = []
    lines = text.split('\n')

    for line in lines:
        # 提取时间戳后面的内容
        match = re.search(r'\[.*\]\s*(.*)', line)
        if match:
            content = match.group(1).strip()

            # --- 过滤逻辑开始 ---
            # A. 检查是否为空
            if not content:
                continue

            # B. 检查是否包含黑名单关键词
            if any(k in content.upper() for k in exclude_keywords): # 转大写匹配，防止漏掉 isrc
                continue

            # C. 检查是否匹配 ISRC 正则特征
            if re.search(isrc_pattern, content):
                continue

            # --- 过滤逻辑结束 ---

            # 内部空格换逗号，去除多余空白
            clean_content = re.sub(r'\s+', ' ', content).replace(" ", "，")
            pure_lyrics_list.append(clean_content)

    # 用句号连接
    if not pure_lyrics_list: return ""
    return "。".join(pure_lyrics_list) + "。"

# --- 制作信息提取 ---
def get_credit_info(raw_text):
    """从原始文本中提取 作词/作曲/编曲"""
    info = {"作词": "", "作曲": "", "编曲": ""}
    # 兼容带时间戳和不带时间戳的情况
    for key in info.keys():
        pattern = rf"{key}\s*[:：]\s*([^\]\n]+)"
        match = re.search(pattern, raw_text)
        if match:
            # 清理掉可能残余的括号或空格
            info[key] = match.group(1).strip()
    return info

# --- API 请求函数 ---
def get_lyrics_by_api(song_id):
    api_url = f"https://music.163.com/api/song/lyric?id={song_id}&lv=1&kv=1&tv=-1"
    headers = {"User-Agent": "Mozilla/5.0"}

    try:
        response = requests.get(api_url, headers=headers, timeout=10)
        response.raise_for_status()
        data = response.json()
        return data.get('lrc', {}).get('lyric', "")
    except Exception as e:
        print(f"ID {song_id} 获取失败: {e}")
        return ""

# --- 主逻辑封装 ---
def process_single_song(song_id, song_name):
    """处理单首歌曲：下载 -> 解析 -> 组装字典"""
    raw_lyric = get_lyrics_by_api(song_id)

    # 提取制作人信息
    credits = get_credit_info(raw_lyric)
    # 提取并清洗歌词
    formatted_lyric = clean_and_format_lyrics(raw_lyric)

    has_lyric = 1
    if not raw_lyric:
        has_lyric = 0
        formatted_lyric = ""
    # 组装结果
    return {
        "歌名": song_name,
        "song_id": song_id,
        "has_lyric": has_lyric,
        "作词": credits["作词"],
        "作曲": credits["作曲"],
        "编曲": credits["编曲"],
        "歌词": formatted_lyric
    }

# --- 文件保存（增量更新） ---
def save_to_json_list(file_path, song_data):
    """以列表形式保存所有歌曲，避免字典 key 覆盖的问题"""
    data_list = []
    if os.path.exists(file_path):
        with open(file_path, 'r', encoding='utf-8') as f:
            try:
                data_list = json.load(f)
                if not isinstance(data_list, list): data_list = []
            except:
                data_list = []

    data_list.append(song_data)

    with open(file_path, 'w', encoding='utf-8') as f:
        json.dump(data_list, f, ensure_ascii=False, indent=4)

In [ ]:
# 已采集歌词
song_had_lyric = df_raw[df_raw['has_lyric'].notnull()]['song_name'].unique().tolist()
len(song_had_lyric) 

In [ ]:
file_path = 'output/mayday_lyric_temp.json'
# 遍历df_df_unique每一行
# 每次运行前删除file_path文件
# 运行结束后，将json内容手动复制到主文件
for index, row in df_unique.iterrows():
    song_id = str(row['song_id'])
    song_name = row['song_name']
    # 判断是否已采集
    if song_name not in song_had_lyric:
        print(song_name)
        single_res = process_single_song(song_id, song_name)
        save_to_json_list(file_path, single_res)
        time.sleep(2)

# 词频与词性分析

In [33]:
word_to_fix = {
    '阮': 'r',
    '袂': 'v'
}

In [34]:
def process_lyrics_with_jieba(text):
    # 1. 词性标注与分词
    # jieba.posseg 会同时返回词和词性
    words_with_pos = pseg.cut(text)

    
    # 2. 过滤无意义字符（标点、空格、单字符停用词）
    filtered_data = []
    for word, pos in words_with_pos:
        # 排除标点符号（x表示标点）及空白字符
        if pos != 'x' and len(word.strip()) > 0:
            if word in word_to_fix:
                filtered_data.append((word, word_to_fix[word]))
            else:
                filtered_data.append((word, pos))
    
    # 3. 统计词频
    word_counts = Counter([item[0] for item in filtered_data])
    
    # 4. 汇总信息 (词, 词性, 频数)
    # 我们以词为 Key，存储词性
    word_pos_map = {word: pos for word, pos in filtered_data}
    
    # 排序：按词频从高到低
    sorted_results = []
    for word, count in word_counts.most_common():
        sorted_results.append({
            "word": word,
            "pos": word_pos_map[word],
            "freq": count # 词频
        })
    
    return sorted_results

In [35]:
# 读取歌词文件
with open("output/mayday_lyric_260124.json", 'r') as f:
    lyric_data = json.load(f)

In [36]:
lyric_words_dict = {}
for i in lyric_data:
    if i:
        lyric_words_dict[i['song_id']] = process_lyrics_with_jieba(i['歌词'])
lyric_words_dict

{'386925': [{'word': '我', 'pos': 'r', 'freq': 18},
  {'word': '好想', 'pos': 'v', 'freq': 18},
  {'word': '那么', 'pos': 'r', 'freq': 12},
  {'word': '多', 'pos': 'm', 'freq': 12},
  {'word': '的', 'pos': 'uj', 'freq': 12},
  {'word': '这个', 'pos': 'r', 'freq': 9},
  {'word': '世界', 'pos': 'n', 'freq': 9},
  {'word': '飞', 'pos': 'v', 'freq': 9},
  {'word': '逃离', 'pos': 'v', 'freq': 8},
  {'word': '疯狂', 'pos': 'a', 'freq': 8},
  {'word': '你', 'pos': 'r', 'freq': 7},
  {'word': '了', 'pos': 'ul', 'freq': 6},
  {'word': '是', 'pos': 'v', 'freq': 6},
  {'word': '苦', 'pos': 'a', 'freq': 4},
  {'word': '累', 'pos': 'v', 'freq': 4},
  {'word': '莫名', 'pos': 'v', 'freq': 4},
  {'word': '如果', 'pos': 'c', 'freq': 4},
  {'word': '发现', 'pos': 'v', 'freq': 4},
  {'word': '也别', 'pos': 'd', 'freq': 4},
  {'word': '将', 'pos': 'd', 'freq': 4},
  {'word': '挽回', 'pos': 'v', 'freq': 4},
  {'word': '泪水', 'pos': 'n', 'freq': 3},
  {'word': '后悔', 'pos': 'v', 'freq': 2},
  {'word': '多么', 'pos': 'r', 'freq': 2},
  {'word'

In [37]:
rows = []
for song_id, word_list in lyric_words_dict.items():
    for item in word_list:
        # 创建新字典，保留原始数据并加入歌曲ID列
        new_row = {
            'song_id': song_id,
            'word': item['word'],
            'pos': item['pos'],
            'freq': item['freq']
        }
        rows.append(new_row)

# 3. 转换为 DataFrame
df_word = pd.DataFrame(rows)
df_word

,song_id,word,pos,freq
0,386925,我,r,18
1,386925,好想,v,18
2,386925,那么,r,12
3,386925,多,m,12
4,386925,的,uj,12
...,...,...,...,...
13380,28181110,长大,v,1
13381,28181110,难道,d,1
13382,28181110,人,n,1
13383,28181110,必经,d,1


In [38]:
# 合并
# 1. 确保 df_word 的 song_id 是字符串
df_word['song_id'] = df_word['song_id'].astype(str)

# 2. 确保 df_unique 的 song_id 是字符串（并去掉可能存在的空格）
df_unique['song_id'] = df_unique['song_id'].astype(str).str.strip()

# 3. 执行合并
df_merged = df_word.merge(df_unique, on='song_id', how='left')

# 4. 删除空值
df_merged = df_merged.dropna()

df_merged

,song_id,word,pos,freq,album_order,album_id,album_name,album_type,release_date,song_order,song_name,album_fixed,has_lyric,is_duplicate,song_name_fixed
0,386925,我,r,18,1.0,38315.0,第一张创作专辑,录音室专辑,1999-07-07,1.0,疯狂世界,第一张创作专辑,1.0,0.0,疯狂世界
1,386925,好想,v,18,1.0,38315.0,第一张创作专辑,录音室专辑,1999-07-07,1.0,疯狂世界,第一张创作专辑,1.0,0.0,疯狂世界
2,386925,那么,r,12,1.0,38315.0,第一张创作专辑,录音室专辑,1999-07-07,1.0,疯狂世界,第一张创作专辑,1.0,0.0,疯狂世界
3,386925,多,m,12,1.0,38315.0,第一张创作专辑,录音室专辑,1999-07-07,1.0,疯狂世界,第一张创作专辑,1.0,0.0,疯狂世界
4,386925,的,uj,12,1.0,38315.0,第一张创作专辑,录音室专辑,1999-07-07,1.0,疯狂世界,第一张创作专辑,1.0,0.0,疯狂世界
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13380,28181110,长大,v,1,11.0,2740205.0,步步 自选作品辑,精选辑,2013-12-30,19.0,盛夏光年,步步 自选作品辑,1.0,0.0,盛夏光年
13381,28181110,难道,d,1,11.0,2740205.0,步步 自选作品辑,精选辑,2013-12-30,19.0,盛夏光年,步步 自选作品辑,1.0,0.0,盛夏光年
13382,28181110,人,n,1,11.0,2740205.0,步步 自选作品辑,精选辑,2013-12-30,19.0,盛夏光年,步步 自选作品辑,1.0,0.0,盛夏光年
13383,28181110,必经,d,1,11.0,2740205.0,步步 自选作品辑,精选辑,2013-12-30,19.0,盛夏光年,步步 自选作品辑,1.0,0.0,盛夏光年


In [39]:
df_merged.to_csv('output/mayday_lyric_word.csv', index=False)

In [ ]:
with open('lyric_temp.json', 'w', encoding='utf-8') as f:
    json.dump(lyric_words_dict, f, ensure_ascii=False, indent=4)

In [ ]:
def analyze_lyrics_counts(lyric_words_dict):
    # 1. 加载数据
    # with open(file_path, 'r', encoding='utf-8') as f:
    #     data = json.load(f)
    data = lyric_words_dict
    
    # 2. 初始化统计
    # stats 的结构: {(词, 词性): [出现次数, 频数累计]}
    stats = defaultdict(lambda: [0, 0])
    total_frequency_sum = 0 # 记录所有频数之和
    
    # 3. 遍历统计
    for song_id, word_list in data.items():
        for item in word_list:
            word = item['词']
            pos = item['词性']
            freq = item['频数']
            
            # 唯一标识符
            key = (word, pos)
            
            # “次数”：每在列表中出现一次，计数 +1
            stats[key][0] += 1
            # “频数”：累加歌曲中的具体数值
            stats[key][1] += freq
            # “频数之和”：全局累加
            total_frequency_sum += freq

    # 4. 转换并排序
    final_list = []
    for (word, pos), (occurrence, total_freq) in stats.items():
        final_list.append({
            "词": word,
            "词性": pos,
            "出现次数": occurrence, # 在 JSON 列表里出现的次数
            "累计频数": total_freq   # 频数数值的加总
        })
    
    # 按“累计频数”降序排列
    final_list.sort(key=lambda x: x['累计频数'], reverse=True)
    
    return final_list, total_frequency_sum

In [ ]:
analyze_lyrics_counts(lyric_words_dict)

## 整体词频统计

# 歌曲热度

In [ ]:
songs = df_unique['song_name'].to_list()

In [ ]:
len(songs)

In [ ]:
songs_heat_index = [
    {"歌名": "突然好想你", "排名": 1},
    {"歌名": "倔强", "排名": 2},
    {"歌名": "温柔", "排名": 3},
    {"歌名": "后来的我们", "排名": 4},
    {"歌名": "知足", "排名": 5},
    {"歌名": "你不是真正的快乐", "排名": 6},
    {"歌名": "恋爱ing", "排名": 7},
    {"歌名": "我不愿让你一个人", "排名": 8},
    {"歌名": "干杯", "排名": 9},
    {"歌名": "志明与春娇", "排名": 10},
    {"歌名": "如烟", "排名": 11},
    {"歌名": "成名在望", "排名": 12},
    {"歌名": "派对动物", "排名": 13},
    {"歌名": "OAOA (现在就是永远)", "排名": 14},
    {"歌名": "仓颉", "排名": 15},
    {"歌名": "顽固", "排名": 16},
    {"歌名": "疯狂世界", "排名": 17},
    {"歌名": "诺亚方舟", "排名": 18},
    {"歌名": "人生海海", "排名": 19},
    {"歌名": "星空", "排名": 20},
    {"歌名": "天使", "排名": 21},
    {"歌名": "最重要的小事", "排名": 22},
    {"歌名": "憨人", "排名": 23},
    {"歌名": "孙悟空", "排名": 24},
    {"歌名": "一颗苹果", "排名": 25},
    {"歌名": "终结孤单", "排名": 26},
    {"歌名": "拥抱", "排名": 27},
    {"歌名": "恒星的恒心", "排名": 28},
    {"歌名": "时光机", "排名": 29},
    {"歌名": "我心中尚未崩坏的地方", "排名": 30},
    {"歌名": "让我照顾你", "排名": 31},
    {"歌名": "彩虹", "排名": 32},
    {"歌名": "爱情万岁", "排名": 33},
    {"歌名": "轧车", "排名": 34},
    {"歌名": "垃圾车", "排名": 35},
    {"歌名": "有些事现在不做 一辈子都不会做了", "排名": 36},
    {"歌名": "纯真", "排名": 37},
    {"歌名": "温柔 (还你自由版)", "排名": 38},
    {"歌名": "出头天", "排名": 39},
    {"歌名": "生命有一种绝对", "排名": 40},
    {"歌名": "风若吹", "排名": 41},
    {"歌名": "黑白讲", "排名": 42},
    {"歌名": "为什么 (今日的爱情)", "排名": 43},
    {"歌名": "雨眠", "排名": 44},
    {"歌名": "罗密欧与茱丽叶", "排名": 45},
    {"歌名": "好不好", "排名": 46},
    {"歌名": "借问众神明", "排名": 47},
    {"歌名": "永远的永远", "排名": 48},
    {"歌名": "嘿！我要走了", "排名": 49},
    {"歌名": "透露", "排名": 50},
    {"歌名": "生活", "排名": 51},
    {"歌名": "爱情的模样", "排名": 52},
    {"歌名": "I Love You 无望", "排名": 53},
    {"歌名": "HoSee", "排名": 54},
    {"歌名": "啾啾啾", "排名": 55},
    {"歌名": "雌雄同体", "排名": 56},
    {"歌名": "阿姆斯壮", "排名": 57},
    {"歌名": "而我知道", "排名": 58},
    {"歌名": "赌神", "排名": 59},
    {"歌名": "别惹我", "排名": 60},
    {"歌名": "九号球", "排名": 61},
    {"歌名": "武装", "排名": 62},
    {"歌名": "我们 (时时刻刻)", "排名": 63},
    {"歌名": "在这一秒", "排名": 64},
    {"歌名": "王子面", "排名": 65},
    {"歌名": "小时候", "排名": 66},
    {"歌名": "小护士", "排名": 67},
    {"歌名": "约翰蓝侬", "排名": 68},
    {"歌名": "回来吧", "排名": 69},
    {"歌名": "错错错", "排名": 70},
    {"歌名": "晚安，地球人", "排名": 71},
    {"歌名": "超人", "排名": 72},
    {"歌名": "神的孩子都在跳舞", "排名": 73},
    {"歌名": "圣诞夜惊魂", "排名": 74},
    {"歌名": "垃圾车(朋友版)", "排名": 75},
    {"歌名": "前传", "排名": 76},
    {"歌名": "为爱而生", "排名": 77},
    {"歌名": "我又初恋了", "排名": 78},
    {"歌名": "香水", "排名": 79},
    {"歌名": "摩托车日记", "排名": 80},
    {"歌名": "快乐很伟大", "排名": 81},
    {"歌名": "忘词", "排名": 82},
    {"歌名": "宠上天", "排名": 83},
    {"歌名": "米老鼠", "排名": 84},
    {"歌名": "一千个世纪", "排名": 85},
    {"歌名": "胎音", "排名": 86},
    {"歌名": "生存以上 生活以下", "排名": 87},
    {"歌名": "爆肝", "排名": 88},
    {"歌名": "噢买尬", "排名": 89},
    {"歌名": "春天的呐喊", "排名": 90},
    {"歌名": "夜访吸血鬼", "排名": 91},
    {"歌名": "后青春期的诗", "排名": 92},
    {"歌名": "笑忘歌", "排名": 93},
    {"歌名": "洗衣机", "排名": 94},
    {"歌名": "三个傻瓜", "排名": 95},
    {"歌名": "歪腰", "排名": 96},
    {"歌名": "2012", "排名": 97},
    {"歌名": "第二人生", "排名": 98},
    {"歌名": "明日", "排名": 99},
    {"歌名": "OAOA (丢掉名字性别)", "排名": 100},
    {"歌名": "如果我们不曾相遇", "排名": 101},
    {"歌名": "好好 (想把你写成一首歌)", "排名": 102},
    {"歌名": "兄弟", "排名": 103},
    {"歌名": "人生有限公司", "排名": 104},
    {"歌名": "最好的一天", "排名": 105},
    {"歌名": "少年他的奇幻漂流", "排名": 106},
    {"歌名": "终于结束的起点", "排名": 107},
    {"歌名": "任意门", "排名": 108},
    {"歌名": "转眼", "排名": 109},
    {"歌名": "牙关", "排名": 110},
    {"歌名": "乱世浮生", "排名": 111},
    {"歌名": "听不到", "排名": 112},
    {"歌名": "金多虾", "排名": 113},
    {"歌名": "麦来乱", "排名": 114},
    {"歌名": "OK 啦", "排名": 115},
    {"歌名": "垃圾车 (朋友版)", "排名": 116},
    {"歌名": "未来 (Sailing With Me)", "排名": 117},
    {"歌名": "咸鱼", "排名": 118},
    {"歌名": "明白", "排名": 119},
    {"歌名": "心中无别人", "排名": 120},
    {"歌名": "有你的将来", "排名": 121},
    {"歌名": "叫我第一名", "排名": 122},
    {"歌名": "能不能不要说", "排名": 123},
    {"歌名": "相信", "排名": 124},
    {"歌名": "Ok啦", "排名": 125},
    {"歌名": "候鸟", "排名": 126},
    {"歌名": "轻功", "排名": 127},
    {"歌名": "反而", "排名": 128}
]

In [ ]:
# 为df_unique添加heat_index列
songs_heat_dict = {}
for i in songs_heat_index:
    songs_heat_dict[i['歌名']] = i['排名']
songs_heat_dict

In [ ]:
df_unique['heat_index'] = df_unique['song_name'].map(songs_heat_dict).fillna(0)

In [ ]:
df_unique